In [77]:
# Cell 1: Install dependencies (run once)
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html -q

In [9]:
# Cell 2: Imports and Configuration
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.data import Data, Batch
import os
import pandas as pd
from tqdm.auto import tqdm
import random
import time

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Configuration
# Get the directory where this notebook is located
import pathlib
NOTEBOOK_DIR = pathlib.Path().absolute()
BASE_DIR = os.path.join(str(NOTEBOOK_DIR), 'Scale_Results')
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Results will be saved to: {BASE_DIR}")

# Resolution reduction factor (K=4 means 4x smaller in each dimension)
K = 4

# Original dimensions divided by K
ORIGINAL_X, ORIGINAL_Y, ORIGINAL_Z = 681 // K, 344 // K, max(12 // K, 3)

SCALES = [
    ('original', 1, 1, 1), ('2x1y1z', 2, 1, 1), ('1x2y1z', 1, 2, 1),
    ('1x1y2z', 1, 1, 2), ('2x2y1z', 2, 2, 1), ('1x2y2z', 1, 2, 2),
    ('2x2y2z', 2, 2, 2), ('3x1y1z', 3, 1, 1), ('3x2y1z', 3, 2, 1),
    ('3x1y2z', 3, 1, 2), ('3x2y2z', 3, 2, 2)
]

TEXTURES = ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']
Z_FIXED, MAX_RADIUS, STEP_SIZE = max(5 // K, 1), max(10 // K, 6), max(5 // K, 2)

# Match tolerance for comparing model output with GT
# Increase tolerance to improve matching (helps with coordinate alignment)
MATCH_TOLERANCE = max(MAX_RADIUS, 4)  # voxels (at least 4 for better matching)

# ============================================================
# MODEL SCORE FILTERING OPTIONS (to improve Precision)
# ============================================================
# Option 1: Fixed threshold (None = disabled)
MODEL_SCORE_THRESHOLD = None  # e.g., 0.5 = only score >= 0.5

# Option 2: Percentile-based threshold (None = disabled)
# Using percentile: 75 = only top 25% (score >= 75th percentile)
# Fixed percentile doesn't adapt well to different score distributions
MODEL_SCORE_PERCENTILE = None  # e.g., 60, 65, 70, 75, 80, 85, 90

# Option 3: Mean + std multiplier (RECOMMENDED - adapts to each scale)
# threshold = mean + (multiplier * std)
# This adapts to each scale's score distribution automatically
MODEL_SCORE_STD_MULTIPLIER = 0.3  # e.g., 0.0 = mean, 0.3 = mean+0.3*std, 0.5 = mean+0.5*std

# Option 4: Top percentage (None = disabled, applied after threshold)
MODEL_TOP_PERCENTAGE = None  # e.g., 0.7 = only top 70%

# Option 5: Match model predictions count to GT count (for better balance)
MODEL_MATCH_GT_COUNT = False  # Set to False to keep all filtered predictions (better recall)

# Note: If multiple options are enabled:
# 1. First, threshold/percentile/std is applied
# 2. Then, top percentage is applied

print(f"\nResolution reduction factor: K={K}")
print(f"Match tolerance: {MATCH_TOLERANCE} voxels")
if MODEL_SCORE_THRESHOLD is not None:
    print(f"Model score threshold: {MODEL_SCORE_THRESHOLD}")
if MODEL_SCORE_PERCENTILE is not None:
    print(f"Model score percentile: {MODEL_SCORE_PERCENTILE}th")
if MODEL_SCORE_STD_MULTIPLIER is not None:
    print(f"Model score threshold: mean + {MODEL_SCORE_STD_MULTIPLIER}*std")
if MODEL_TOP_PERCENTAGE is not None:
    print(f"Model top percentage: {MODEL_TOP_PERCENTAGE*100:.0f}%")
if MODEL_MATCH_GT_COUNT:
    print(f"Model predictions will be matched to GT count for balance")
print(f"Effective dimensions: {ORIGINAL_X}x{ORIGINAL_Y}x{ORIGINAL_Z}")
print(f"Textures: {TEXTURES}")
print(f"Scales: {len(SCALES)}")
print(f"Total experiments: {len(TEXTURES) * len(SCALES)}")

Device: cpu
Results will be saved to: d:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale_Results

Resolution reduction factor: K=4
Match tolerance: 6 voxels
Model score threshold: mean + 0.3*std
Effective dimensions: 170x86x3
Textures: ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']
Scales: 11
Total experiments: 55


In [10]:
# Cell 3: Texture Generation Functions (Vectorized)

def generate_sinusoid_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Sinusoidal wave patterns - vectorized for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    # Scale parameters based on dimensions
    y_center = y_dim // 3
    strip_size = max(y_dim // 15, 3)
    
    # Scale sine parameters proportionally
    sine_params = [
        (0, y_dim // 3, x_dim),      # amplitude, period scaled
        (1, y_dim // 6, x_dim // 2),
        (2, y_dim // 2, x_dim * 3 // 4)
    ]
    
    x_range = np.arange(x_dim)
    for channel, amplitude, period in sine_params:
        if period == 0:
            period = 1
        y_sine = y_center + amplitude * np.sin(2 * np.pi * x_range / period)
        for z in range(z_dim):
            for x in range(x_dim):
                y_c = int(round(y_sine[x]))
                y_start, y_end = max(0, y_c - strip_size//2), min(y_dim, y_c + strip_size//2 + 1)
                for y in range(y_start, y_end):
                    data[channel, np.random.randint(0, 11), z, y, x] = 1
    
    # Channel 3: horizontal line
    y_start, y_end = max(0, y_center - strip_size//2), min(y_dim, y_center + strip_size//2 + 1)
    for z in range(z_dim):
        for x in range(x_dim):
            for y in range(y_start, y_end):
                data[3, np.random.randint(0, 11), z, y, x] = 1
    return data


def generate_linear_texture(num_channels, num_values, z_dim, y_dim, x_dim, x_scale, y_scale):
    """Linear strip patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    
    # Scale parameters for low resolution
    base_offset = x_dim // 18
    base_width = max(x_dim // 140, 2)
    
    for i in range(3):
        for z in range(z_dim):
            x_start = max(0, int((base_offset + base_offset*i) * x_scale))
            x_end = min(x_dim, int((base_offset + base_width + base_offset*i) * x_scale))
            for y in range(y_dim):
                for x in range(x_start, x_end):
                    data[0, np.random.randint(6, 11), z, y, x] = 1
            
            y_base = y_dim // 20
            y_start = max(0, int((y_base + y_base*i) * y_scale))
            y_end = min(y_dim, int((y_base + base_width + y_base*i) * y_scale))
            for y in range(y_start, y_end):
                for x in range(x_dim):
                    data[1, np.random.randint(6, 11), z, y, x] = 1
            
            strip_w = max(int(2 * max(x_scale, y_scale)), 1)
            diag_offset = x_dim // 12
            for c in [int(-diag_offset * x_scale), 0]:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y - x - c) <= strip_w:
                            data[2, np.random.randint(6, 11), z, y, x] = 1
            for d in [int(diag_offset * y_scale), int(2 * diag_offset * y_scale)]:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y + x - d) <= strip_w:
                            data[3, np.random.randint(6, 11), z, y, x] = 1
    return data


def generate_olympic_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Olympic rings pattern - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    
    # Scale circles based on dimensions
    circles = [
        (0, x_dim * 3 // 10, y_dim * 4 // 10, min(x_dim, y_dim) // 4),
        (1, x_dim * 6 // 10, y_dim * 4 // 10, min(x_dim, y_dim) // 4),
        (2, x_dim * 45 // 100, y_dim * 7 // 10, min(x_dim, y_dim) // 4),
        (3, x_dim * 5 // 10, y_dim * 9 // 10, min(x_dim, y_dim) // 3)
    ]
    stripe_width = max(min(x_dim, y_dim) // 20, 2)
    
    for channel, cx, cy, radius in circles:
        inner_r, outer_r = max(radius - stripe_width, 1), radius
        for z in range(z_dim):
            for y in range(y_dim):
                for x in range(x_dim):
                    dist = np.sqrt((x-cx)**2 + (y-cy)**2)
                    if inner_r <= dist <= outer_r:
                        data[channel, np.random.randint(0, 11), z, y, x] = 1
    return data


def generate_oval_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Oval/ellipse patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    cx, cy = x_dim // 2, y_dim // 2
    x_stretch = 1.5
    
    # Scale radii based on dimensions
    min_dim = min(x_dim, y_dim)
    inner_r1, outer_r1 = min_dim // 4, min_dim // 2
    inner_r2, outer_r2 = min_dim * 4 // 10, min_dim * 55 // 100
    
    for z in range(z_dim):
        for y in range(y_dim):
            for x in range(x_dim):
                dx, dy = (x - cx) / x_stretch, y - cy
                dist = np.sqrt(dx**2 + dy**2)
                if inner_r1 <= dist <= outer_r1:
                    data[0, np.random.randint(6, 11), z, y, x] = 1
                if inner_r2 <= dist <= outer_r2:
                    data[1, np.random.randint(6, 11), z, y, x] = 1
    
    z_center = z_dim // 2
    cloud_r = (inner_r1 + outer_r1) // 2
    cloud_spread = max(min_dim // 10, 2)
    z_spread = max(z_dim // 4, 1)
    
    for _ in range(5):
        angle = np.random.uniform(0, 2*np.pi)
        r = np.random.uniform(cloud_r * 0.9, cloud_r * 1.1)
        cloud_cx = int(np.clip(cx + r * x_stretch * np.cos(angle), cloud_spread, x_dim - cloud_spread - 1))
        cloud_cy = int(np.clip(cy + r * np.sin(angle), cloud_spread, y_dim - cloud_spread - 1))
        for _ in range(max(20, min_dim // 5)):
            rx = np.random.randint(-cloud_spread, cloud_spread + 1)
            ry = np.random.randint(-cloud_spread, cloud_spread + 1)
            rz = np.random.randint(-z_spread, z_spread + 1)
            px, py, pz = cloud_cx + rx, cloud_cy + ry, z_center + rz
            if 0 <= px < x_dim and 0 <= py < y_dim and 0 <= pz < z_dim:
                data[2, np.random.randint(6, 11), pz, py, px] = 1
                data[3, np.random.randint(6, 11), pz, py, px] = 1
    return data


def generate_colonies_texture(num_channels, num_values, z_dim, y_dim, x_dim, x_scale, y_scale):
    """Colony cloud patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    min_dim = min(x_dim, y_dim)
    max_radius = max(min_dim // 8, 3)
    cloud_points = max(10, min_dim // 10)
    
    def add_cloud(channel, cx, cy, z, radius):
        for _ in range(cloud_points):
            angle = np.random.uniform(0, 2*np.pi)
            r = np.random.uniform(0, radius)
            px = int(np.clip(cx + r*np.cos(angle), 0, x_dim-1))
            py = int(np.clip(cy + r*np.sin(angle), 0, y_dim-1))
            data[channel, np.random.randint(6, 11), z, py, px] = 1
    
    # Scale step sizes
    arc_steps = max(x_dim // 8, 6)
    sine_step = max(x_dim // 8, 4)
    diag_step = max(x_dim // 6, 5)
    
    for z in range(z_dim):
        for t in np.linspace(0, 1, arc_steps):
            arc_x = int(t * (x_dim - 1))
            arc_y = int(t * (y_dim - 1) + 0.3 * (y_dim - 1) * np.sin(t * np.pi))
            arc_y = int(np.clip(arc_y, 0, y_dim-1))
            add_cloud(0, arc_x, arc_y, z, np.random.uniform(max_radius // 3, max_radius))
        
        for x in range(0, x_dim, sine_step):
            y_center = y_dim // 3
            y_amp = y_dim // 6
            y = int(y_center + y_amp * np.sin(2*np.pi*x/x_dim))
            y = int(np.clip(y, 0, y_dim-1))
            add_cloud(1, x, y, z, np.random.uniform(max_radius // 3, max_radius))
        
        diag_offsets = [y_dim // 5, y_dim * 2 // 5]
        for d in diag_offsets:
            for x in range(0, x_dim, diag_step):
                y = -x + int(d * y_scale)
                if 0 <= y < y_dim:
                    add_cloud(3, x, y, z, np.random.uniform(max_radius // 3, max_radius))
    return data


def create_groundtruth(texture_type, scale_name, x_scale, y_scale, z_scale, output_dir):
    """Create ground truth with specified texture and scale"""
    num_channels, num_values = 4, 11
    x_dim = ORIGINAL_X * x_scale
    y_dim = ORIGINAL_Y * y_scale  
    z_dim = ORIGINAL_Z * z_scale
    local_x_scale, local_y_scale = x_dim / 172, y_dim / 87
    
    if texture_type == 'sinusoid':
        data = generate_sinusoid_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'linear':
        data = generate_linear_texture(num_channels, num_values, z_dim, y_dim, x_dim, local_x_scale, local_y_scale)
    elif texture_type == 'olympic':
        data = generate_olympic_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'oval':
        data = generate_oval_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'colonies':
        data = generate_colonies_texture(num_channels, num_values, z_dim, y_dim, x_dim, local_x_scale, local_y_scale)
    else:
        raise ValueError(f"Unknown texture: {texture_type}")
    
    filepath = os.path.join(output_dir, f'groundtruth_{texture_type}_{scale_name}.npy')
    np.save(filepath, data)
    return data, filepath

print("Texture generation functions loaded!")

Texture generation functions loaded!


In [11]:
# Cell 4: Subgraph Creation (Optimized)

def create_subgraphs(data, texture_type, scale_name, output_dir):
    """Fast subgraph creation using vectorized operations"""
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    z_idx = min(Z_FIXED, z_dim - 1)
    
    # Pre-compute mask
    intensity_matrix = np.zeros((y_dim, x_dim, num_channels), dtype=np.float32)
    channel_mask = np.zeros((y_dim, x_dim, num_channels), dtype=bool)
    
    for ch in range(num_channels):
        ch_data = data[ch, :, z_idx, :, :]
        mask = ch_data.sum(axis=0) > 0
        intensity_matrix[:, :, ch] = np.where(mask, np.argmax(ch_data, axis=0), 0)
        channel_mask[:, :, ch] = mask
    
    channel_counts = channel_mask.sum(axis=2)
    centers = [(x, y, z_idx) for x in range(0, x_dim, STEP_SIZE) for y in range(0, y_dim, STEP_SIZE)]
    
    all_subgraphs = []
    for cx, cy, cz in centers:
        x_min, x_max = max(0, cx - MAX_RADIUS), min(x_dim, cx + MAX_RADIUS + 1)
        y_min, y_max = max(0, cy - MAX_RADIUS), min(y_dim, cy + MAX_RADIUS + 1)
        
        nodes, positions, active_chs = [], [], []
        for y in range(y_min, y_max):
            for x in range(x_min, x_max):
                if channel_counts[y, x] > 0:
                    dist = np.sqrt((x-cx)**2 + (y-cy)**2)
                    if dist <= MAX_RADIUS:
                        active = np.where(channel_mask[y, x, :])[0].tolist()
                        nodes.append(intensity_matrix[y, x, active])
                        positions.append((x, y, z_idx))
                        active_chs.append(active)
        
        if len(nodes) < 2:
            continue
            
        max_ch = max(len(ch) for ch in active_chs)
        padded = []
        for i, n in enumerate(nodes):
            p = np.zeros(max_ch, dtype=np.float32)
            p[:len(n)] = n
            padded.append(p)
        
        node_features = np.array(padded, dtype=np.float32)
        pos_array = np.array(positions, dtype=np.int32)
        
        # Fast edge creation
        diff = pos_array[:, np.newaxis, :] - pos_array[np.newaxis, :, :]
        dist_matrix = np.sqrt(np.sum(diff**2, axis=2))
        edge_mask = (dist_matrix <= MAX_RADIUS) & (dist_matrix > 0)
        edge_i, edge_j = np.where(edge_mask)
        
        if len(edge_i) == 0:
            continue
        
        edge_weights = dist_matrix[edge_i, edge_j].astype(np.float32)
        
        graph = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=torch.tensor([edge_i, edge_j], dtype=torch.long),
            edge_attr=torch.tensor(edge_weights, dtype=torch.float32),
            center=(cx, cy, cz)
        )
        all_subgraphs.append(graph)
    
    return all_subgraphs

print("Subgraph creation function loaded!")

Subgraph creation function loaded!


In [12]:
# Cell 5: Model Definition (GPU Optimized)

class ContrastiveGAT(nn.Module):
    def __init__(self, in_channels=4, hidden=32, proj_dim=16, heads=4, dropout=0.1, edge_dim=None):
        super().__init__()
        self.edge_dim = edge_dim
        kw = dict(dropout=dropout)
        if edge_dim: kw['edge_dim'] = edge_dim
        
        self.gat1 = GATConv(in_channels, hidden, heads=heads, concat=True, **kw)
        self.gat2 = GATConv(hidden*heads, hidden, heads=heads, concat=True, **kw)
        self.gat3 = GATConv(hidden*heads, hidden, heads=1, concat=False, **kw)
        
        self.norm1 = nn.LayerNorm(hidden*heads)
        self.norm2 = nn.LayerNorm(hidden*heads)
        self.dropout = nn.Dropout(dropout)
        
        self.projection = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, proj_dim)
        )
        self.interaction_head = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden//2, 1)
        )
    
    def forward(self, x, edge_index, edge_attr=None, batch=None):
        ea = edge_attr if self.edge_dim and edge_attr is not None else None
        x = F.elu(self.dropout(self.norm1(self.gat1(x, edge_index, ea))))
        x = F.elu(self.dropout(self.norm2(self.gat2(x, edge_index, ea))))
        x = F.elu(self.gat3(x, edge_index, ea))
        
        if batch is None:
            batch = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)
        
        emb = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch), global_add_pool(x, batch)], dim=1)
        proj = F.normalize(self.projection(emb), dim=1)
        score = self.interaction_head(emb)
        return proj, score


def prepare_graph(graph, target_channels=4):
    x = graph.x.clone()
    if x.shape[1] < target_channels:
        x = torch.cat([x, torch.zeros(x.shape[0], target_channels - x.shape[1])], dim=1)
    elif x.shape[1] > target_channels:
        x = x[:, :target_channels]
    g = Data(x=x, edge_index=graph.edge_index.clone())
    if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
        g.edge_attr = graph.edge_attr.clone()
    if hasattr(graph, 'gt_score'):
        g.gt_score = graph.gt_score
    return g


def graph_augment(graph, target_channels=4):
    g = prepare_graph(graph, target_channels)
    num_nodes = g.x.shape[0]
    mask_n = int(num_nodes * 0.1)
    if mask_n > 0:
        g.x[torch.randperm(num_nodes)[:mask_n]] = 0.0
    return g


def contrastive_loss(z1, z2, temp=0.1):
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    sim = torch.matmul(z1, z2.T) / temp
    labels = torch.arange(z1.shape[0], device=z1.device)
    return (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2


def train_model_gpu(model, graphs, device, epochs=1, batch_size=32, lr=0.01, gradient_accumulation_steps=4):
    """GPU optimized training with gradient accumulation (same as original working code)"""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    target_ch = max(g.x.shape[1] for g in graphs) if graphs else 4
    
    print(f"  Training: {epochs} epochs, batch_size={batch_size}, lr={lr}")
    
    for epoch in range(epochs):
        epoch_losses = []
        optimizer.zero_grad()
        shuffled_graphs = graphs.copy()
        random.shuffle(shuffled_graphs)
        
        for batch_idx, i in enumerate(range(0, len(shuffled_graphs), batch_size)):
            batch_g = shuffled_graphs[i:i+batch_size]
            aug1 = [graph_augment(g, target_ch) for g in batch_g]
            aug2 = [graph_augment(g, target_ch) for g in batch_g]
            
            # Move to GPU
            for g in aug1 + aug2:
                g.x, g.edge_index = g.x.to(device), g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)
            
            try:
                b1, b2 = Batch.from_data_list(aug1), Batch.from_data_list(aug2)
            except:
                continue
            
            z1, p1 = model(b1.x, b1.edge_index, getattr(b1, 'edge_attr', None), b1.batch)
            z2, _ = model(b2.x, b2.edge_index, getattr(b2, 'edge_attr', None), b2.batch)
            
            # Contrastive loss
            loss_contrast = contrastive_loss(z1, z2)
            
            # Supervised regression loss on gt_score
            loss_reg = torch.tensor(0.0, device=device)
            if hasattr(b1, 'gt_score'):
                try:
                    gt = b1.gt_score.to(device).float()
                    pred = p1.view(-1)
                    if gt.std() > 0:
                        gt = (gt - gt.mean()) / (gt.std() + 1e-8)
                    if pred.std() > 0:
                        pred = (pred - pred.mean()) / (pred.std() + 1e-8)
                    loss_reg = F.mse_loss(pred, gt)
                except:
                    pass
            
            # Combined loss with gradient accumulation
            loss = (loss_contrast + 0.5 * loss_reg) / gradient_accumulation_steps
            loss.backward()
            
            epoch_losses.append(loss.item() * gradient_accumulation_steps)
            
            # Update weights every gradient_accumulation_steps
            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            # Cleanup
            del z1, z2, b1, b2, aug1, aug2
            if device.type == 'cuda':
                torch.cuda.empty_cache()
        
        # Final update for remaining batches
        if len(shuffled_graphs) // batch_size % gradient_accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        avg_loss = np.mean(epoch_losses) if epoch_losses else 0.0
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"    Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")
    
    return model

print("Model and training functions loaded!")

Model and training functions loaded!


In [13]:
# Cell 6: Coordinate Extraction & Accuracy (GPU Batch Processing)
# ============================================================
# GT = INTERSECTION VOXELS (where 2+ channels overlap)
# This is INDEPENDENT of graph centers - true intersection locations
# ============================================================


def extract_gt_intersection_coords(data):
    """Extract Ground Truth as INTERSECTION voxels (where 2+ channels overlap).
    
    Algorithm:
        1. For each voxel (z, y, x), check how many channels have value > 0
        2. If 2 or more channels are active at that voxel → it's a GT intersection
        3. Return ALL intersection voxel coordinates
    
    This is INDEPENDENT of graph centers - it's the true intersection locations.
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    
    # channel_active[c, z, y, x] = True if channel c has any value > 0 at that voxel
    channel_active = (data > 0).any(axis=1)  # (C, Z, Y, X)
    
    # Count how many channels are active at each voxel
    channel_count = channel_active.sum(axis=0)  # (Z, Y, X) - values 0, 1, 2, 3, 4
    
    # GT = voxels where 2+ channels overlap (intersection)
    intersection_mask = channel_count >= 2
    
    # Get coordinates of all intersection voxels
    z_indices, y_indices, x_indices = np.where(intersection_mask)
    
    gt_coords = []
    for z, y, x in zip(z_indices, y_indices, x_indices):
        gt_coords.append({'x': int(x), 'y': int(y), 'z': int(z)})
    
    return gt_coords


def attach_gt_scores(data, graphs):
    """Compute GT score based on pattern count (for training) - SAME AS OLD CODE."""
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    
    # pattern_mask[z, y, x] = True if any channel / value > 0 at that voxel
    pattern_mask = (data > 0).any(axis=1).any(axis=0)  # (Z, Y, X)
    
    for g in graphs:
        cx, cy, cz = int(g.center[0]), int(g.center[1]), int(g.center[2])
        cz = np.clip(cz, 0, z_dim-1)
        
        x_min, x_max = max(0, cx-MAX_RADIUS), min(x_dim, cx+MAX_RADIUS+1)
        y_min, y_max = max(0, cy-MAX_RADIUS), min(y_dim, cy+MAX_RADIUS+1)
        
        yy, xx = np.meshgrid(np.arange(y_min, y_max), np.arange(x_min, x_max), indexing='ij')
        dist_sq = (xx - cx)**2 + (yy - cy)**2
        mask = dist_sq <= MAX_RADIUS**2
        
        # Count pattern voxels within radius (same as extract_gt_coords)
        scores_in_radius = pattern_mask[cz, y_min:y_max, x_min:x_max].astype(np.float32)
        g.gt_score = float((scores_in_radius * mask).sum())
    
    return graphs


def find_top_positions_batch(model, graphs, device, top_k, batch_size=128):
    """Batch GPU inference - returns top_k model predictions sorted by score.
    
    Args:
        model: trained model
        graphs: list of graphs
        device: cpu or cuda
        top_k: number of top predictions to return
        batch_size: batch size for inference
    
    Returns:
        List of top_k predictions (sorted by score)
    """
    model.eval()
    target_ch = max(g.x.shape[1] for g in graphs) if graphs else 4
    all_scores = []
    
    with torch.no_grad():
        for i in range(0, len(graphs), batch_size):
            batch_g = graphs[i:i+batch_size]
            prepared = [prepare_graph(g, target_ch) for g in batch_g]
            
            for g in prepared:
                g.x, g.edge_index = g.x.to(device), g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)
            
            try:
                batch = Batch.from_data_list(prepared)
                _, scores = model(batch.x, batch.edge_index, getattr(batch, 'edge_attr', None), batch.batch)
                scores = scores.cpu().numpy().flatten()
                
                for j, g in enumerate(batch_g):
                    all_scores.append({
                        'x': g.center[0], 
                        'y': g.center[1], 
                        'z': g.center[2], 
                        'score': float(scores[j])
                    })
            except:
                for g in batch_g:
                    all_scores.append({
                        'x': g.center[0], 
                        'y': g.center[1], 
                        'z': g.center[2], 
                        'score': 0.0
                    })
    
    # Sort by score (highest first) and return top_k
    all_scores.sort(key=lambda s: s['score'], reverse=True)
    return all_scores[:top_k]




def compute_accuracy(model_coords, gt_coords, tol=None):
    """Compute accuracy: for each model prediction, check if any GT is within tolerance.
    
    Args:
        model_coords: List of model predicted coordinates
        gt_coords: List of ground truth intersection coordinates
        tol: Match tolerance in voxels (default: MATCH_TOLERANCE)
    
    Returns:
        Dictionary with TP, FP, FN, precision, recall, f1_score, accuracy
    """
    if tol is None:
        tol = MATCH_TOLERANCE
    
    n_model = len(model_coords)
    n_gt = len(gt_coords)
    
    if n_model == 0:
        return {'TP': 0, 'FP': 0, 'FN': n_gt, 'precision': 0, 'recall': 0, 
                'f1_score': 0, 'accuracy_iou': 0, 'model_count': 0, 'gt_count': n_gt}
    
    if n_gt == 0:
        return {'TP': 0, 'FP': n_model, 'FN': 0, 'precision': 0, 'recall': 0,
                'f1_score': 0, 'accuracy_iou': 0, 'model_count': n_model, 'gt_count': 0}
    
    model_pts = np.array([[c['x'], c['y'], c['z']] for c in model_coords], dtype=np.float32)
    gt_pts = np.array([[c['x'], c['y'], c['z']] for c in gt_coords], dtype=np.float32)
    
    # For each model prediction, check if ANY GT is within tolerance
    # Use batched computation for memory efficiency
    TP = 0
    batch_size = 500  # Process model predictions in batches
    
    for i in range(0, n_model, batch_size):
        batch_model = model_pts[i:i+batch_size]
        
        # Compute distances from this batch to all GT points
        diff = batch_model[:, np.newaxis, :] - gt_pts[np.newaxis, :, :]
        dist = np.sqrt(np.sum(diff**2, axis=2))  # (batch_size, n_gt)
        
        # For each model point, check if min distance to any GT <= tolerance
        min_dist = dist.min(axis=1)
        TP += int((min_dist <= tol).sum())
        
        del diff, dist  # Free memory
    
    # FP = model predictions with no GT nearby
    FP = n_model - TP
    
    # FN = GT points not matched by any model prediction
    # (compute which GT points are within tolerance of ANY model prediction)
    matched_gt = np.zeros(n_gt, dtype=bool)
    for i in range(0, n_model, batch_size):
        batch_model = model_pts[i:i+batch_size]
        diff = batch_model[:, np.newaxis, :] - gt_pts[np.newaxis, :, :]
        dist = np.sqrt(np.sum(diff**2, axis=2))
        
        # Mark GT points that are matched by this batch
        matched_gt |= (dist.min(axis=0) <= tol)
        del diff, dist
    
    FN = int((~matched_gt).sum())
    
    # Metrics
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    iou = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0
    
    return {'TP': TP, 'FP': FP, 'FN': FN, 'precision': precision, 'recall': recall, 
            'f1_score': f1, 'accuracy_iou': iou, 'model_count': n_model, 'gt_count': n_gt}


print("Coordinate extraction and accuracy functions loaded!")
print(f"GT = INTERSECTION VOXELS (where 2+ channels overlap)")
print(f"MATCH_TOLERANCE: {MATCH_TOLERANCE} voxels")



Coordinate extraction and accuracy functions loaded!
GT = INTERSECTION VOXELS (where 2+ channels overlap)
MATCH_TOLERANCE: 6 voxels


In [14]:
# Cell 7: Main Execution - Run All Experiments

def print_table(texture_name, results):
    """Print formatted table for one texture"""
    print(f"\n{'='*120}")
    print(f"RESULTS TABLE: {texture_name.upper()}")
    print(f"GT = Intersection voxels (where 2+ channels overlap)")
    print(f"{'='*120}")
    print(f"{'Scale Name':>12} {'Scale':>12} {'Model':>8} {'GT':>8} {'TP':>8} {'FP':>8} {'FN':>8} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Acc':>8}")
    print("-"*120)
    for r in results:
        print(f"{r['Scale Name']:>12} {r['Scale']:>12} {r['Model Points']:>8} {r['GT Points']:>8} {r['TP']:>8} {r['FP']:>8} {r['FN']:>8} {r['Precision']:>10.4f} {r['Recall']:>8.4f} {r['F1 Score']:>8.4f} {r['Accuracy']:>8.4f}")
    print("="*120)


def run_texture(texture_type, device):
    """Run all scales for one texture"""
    print(f"\n{'*'*80}")
    print(f"TEXTURE: {texture_type.upper()}")
    print(f"{'*'*80}")
    
    texture_dir = os.path.join(BASE_DIR, texture_type)
    os.makedirs(texture_dir, exist_ok=True)
    
    results = []
    
    for scale_name, x_s, y_s, z_s in tqdm(SCALES, desc=f"{texture_type}"):
        try:
            # 1. Create ground truth
            data, gt_path = create_groundtruth(texture_type, scale_name, x_s, y_s, z_s, texture_dir)
            
            # 2. Create subgraphs
            graphs = create_subgraphs(data, texture_type, scale_name, texture_dir)
            
            # 3. Filter graphs (use reasonable min_nodes based on K)
            # Original code uses 100, but with K=4 we need fewer
            MIN_NODES = max(10, 100 // (K * K))
            filtered = [g for g in graphs if g.x.shape[0] >= MIN_NODES]
            
            if not filtered:
                print(f"  WARNING {scale_name}: No graphs passed filter (total: {len(graphs)}, min_nodes={MIN_NODES})")
                # Try with lower threshold
                MIN_NODES = 2
                filtered = [g for g in graphs if g.x.shape[0] >= MIN_NODES]
                if not filtered:
                    continue
            
            print(f"  {scale_name}: {len(filtered)} graphs after filter (min_nodes={MIN_NODES})")
            
            # 4. Attach GT scores to graphs for supervised training
            filtered = attach_gt_scores(data, filtered)
            
            # 5. Train model (GPU) - 10 epochs like original
            in_ch = max(g.x.shape[1] for g in filtered)
            edge_dim = 1 if any(hasattr(g, 'edge_attr') and g.edge_attr is not None for g in filtered) else None
            model = ContrastiveGAT(in_ch, 32, 16, 4, 0.1, edge_dim).to(device)
            
            # Subsample for training if too many graphs
            train_graphs = filtered[::2] if len(filtered) > 20000 else filtered
            model = train_model_gpu(model, train_graphs, device, epochs=1, batch_size=32, lr=0.01, gradient_accumulation_steps=4)
            
            # 6. Extract ALL model predictions (sorted by score descending)
            model_coords = find_top_positions_batch(model, filtered, device, len(filtered), batch_size=128)
            
            # Print score statistics and apply filtering
            if model_coords:
                scores = [c['score'] for c in model_coords]
                scores_arr = np.array(scores)
                print(f"    Model Score Statistics:")
                print(f"      Min: {scores_arr.min():.4f}, Max: {scores_arr.max():.4f}")
                print(f"      Mean: {scores_arr.mean():.4f}, Median: {np.median(scores_arr):.4f}, Std: {scores_arr.std():.4f}")
                print(f"      Percentiles: 25th={np.percentile(scores_arr, 25):.4f}, 50th={np.percentile(scores_arr, 50):.4f}, 75th={np.percentile(scores_arr, 75):.4f}, 90th={np.percentile(scores_arr, 90):.4f}, 95th={np.percentile(scores_arr, 95):.4f}")
                
                # Apply threshold filtering
                n_before = len(model_coords)
                threshold = None
                
                # Option 1: Fixed threshold
                if MODEL_SCORE_THRESHOLD is not None:
                    threshold = MODEL_SCORE_THRESHOLD
                    model_coords = [c for c in model_coords if c['score'] >= threshold]
                    print(f"    After fixed threshold ({threshold:.4f}): {len(model_coords)}/{n_before} predictions")
                
                # Option 2: Percentile-based threshold
                elif MODEL_SCORE_PERCENTILE is not None:
                    threshold = np.percentile(scores_arr, MODEL_SCORE_PERCENTILE)
                    model_coords = [c for c in model_coords if c['score'] >= threshold]
                    print(f"    After {MODEL_SCORE_PERCENTILE}th percentile threshold ({threshold:.4f}): {len(model_coords)}/{n_before} predictions")
                
                # Option 3: Mean + std multiplier
                elif MODEL_SCORE_STD_MULTIPLIER is not None:
                    threshold = scores_arr.mean() + (MODEL_SCORE_STD_MULTIPLIER * scores_arr.std())
                    model_coords = [c for c in model_coords if c['score'] >= threshold]
                    print(f"    After mean+{MODEL_SCORE_STD_MULTIPLIER}*std threshold ({threshold:.4f}): {len(model_coords)}/{n_before} predictions")
                
                # Option 4: Top percentage (applied after threshold if any)
                if MODEL_TOP_PERCENTAGE is not None and model_coords:
                    n_top = max(1, int(len(model_coords) * MODEL_TOP_PERCENTAGE))
                    model_coords = model_coords[:n_top]  # Already sorted by score descending
                    print(f"    After top {MODEL_TOP_PERCENTAGE*100:.0f}%: {len(model_coords)} predictions")
            
            # 7. Extract GT = INTERSECTION voxels (where 2+ channels overlap)
            gt_coords = extract_gt_intersection_coords(data)
            
            # 8. Match model predictions count to GT count for better balance
            if MODEL_MATCH_GT_COUNT and model_coords and gt_coords:
                n_match = min(len(model_coords), len(gt_coords))
                model_coords = model_coords[:n_match]  # Keep top n_match predictions (already sorted)
                print(f"    After matching to GT count: {len(model_coords)} model predictions (GT: {len(gt_coords)})")
            
            print(f"    Model predictions: {len(model_coords)}, GT intersections: {len(gt_coords)}")
            
            # 9. Compute accuracy (compare filtered model predictions with GT)
            acc = compute_accuracy(model_coords, gt_coords, MATCH_TOLERANCE)
            
            print(f"    TP={acc['TP']}, FP={acc['FP']}, FN={acc['FN']}")
            print(f"    TP={acc['TP']}, FP={acc['FP']}, FN={acc['FN']}")
            print(f"    Precision={acc['precision']:.4f}, Recall={acc['recall']:.4f}, F1={acc['f1_score']:.4f}")
            
            results.append({
                'Scale Name': scale_name,
                'Scale': f'{x_s}x,{y_s}x,{z_s}x',
                'TP': acc['TP'],
                'FP': acc['FP'],
                'FN': acc['FN'],
                'Model Points': acc['model_count'],
                'GT Points': acc['gt_count'],
                'Precision': acc['precision'],
                'Recall': acc['recall'],
                'F1 Score': acc['f1_score'],
                'Accuracy': acc['accuracy_iou'],
                'Tolerance': MATCH_TOLERANCE
            })
            
            # Cleanup
            del data, graphs, filtered, model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"  ERROR {scale_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Print table
    if results:
        print_table(texture_type, results)
        df = pd.DataFrame(results)
        csv_path = os.path.join(texture_dir, f'results_{texture_type}.csv')
        df.to_csv(csv_path, index=False)
        print(f"  Saved: {csv_path}")
    else:
        print(f"  WARNING: No results for {texture_type}")
    
    return results

print("Execution functions loaded!")

Execution functions loaded!


In [15]:
# Cell 8: RUN ALL EXPERIMENTS

print("="*80)
print("MULTI-TEXTURE SCALE EXPERIMENT - GPU VERSION")
print("="*80)
print(f"\nDevice: {device}")
print(f"Textures: {TEXTURES}")
print(f"Total experiments: {len(TEXTURES) * len(SCALES)}")

start_time = time.time()
all_results = {}

# Run each texture
for texture in TEXTURES:
    all_results[texture] = run_texture(texture, device)

total_time = time.time() - start_time
print(f"\nTotal time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")

MULTI-TEXTURE SCALE EXPERIMENT - GPU VERSION

Device: cpu
Textures: ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']
Total experiments: 55

********************************************************************************
TEXTURE: SINUSOID
********************************************************************************


sinusoid:   0%|          | 0/11 [00:00<?, ?it/s]

  original: 1828 graphs after filter (min_nodes=10)
  Training: 1 epochs, batch_size=32, lr=0.01
    Epoch 1/1 - Loss: 3.092557


sinusoid:   9%|▉         | 1/11 [00:30<05:05, 30.54s/it]

    Model Score Statistics:
      Min: 5.9901, Max: 98.6709
      Mean: 43.0929, Median: 42.3792, Std: 21.2132
      Percentiles: 25th=25.8027, 50th=42.3792, 75th=56.9635, 90th=71.8699, 95th=81.0904
    After mean+0.3*std threshold (49.4569): 690/1828 predictions
    Model predictions: 690, GT intersections: 855
    TP=412, FP=278, FN=0
    TP=412, FP=278, FN=0
    Precision=0.5971, Recall=1.0000, F1=0.7477
  2x1y1z: 3407 graphs after filter (min_nodes=10)
  Training: 1 epochs, batch_size=32, lr=0.01
    Epoch 1/1 - Loss: 3.004732
    Model Score Statistics:
      Min: 15.3446, Max: 493.4620
      Mean: 172.9684, Median: 174.4045, Std: 90.2283
      Percentiles: 25th=111.4885, 50th=174.4045, 75th=210.4172, 90th=300.2920, 95th=340.9959
    After mean+0.3*std threshold (200.0369): 1230/3407 predictions
    Model predictions: 1230, GT intersections: 1698


...

    TP=609, FP=710, FN=0
    TP=609, FP=710, FN=0
    Precision=0.4617, Recall=1.0000, F1=0.6317
  1x1y2z: 1139 graphs after filter (min_nodes=10)
  Training: 1 epochs, batch_size=32, lr=0.01
    Epoch 1/1 - Loss: 3.373340


olympic:  36%|███▋      | 4/11 [02:44<05:01, 43.05s/it]

    Model Score Statistics:
      Min: 17.7283, Max: 183.2435
      Mean: 86.2809, Median: 93.9434, Std: 34.2074
      Percentiles: 25th=63.8937, 50th=93.9434, 75th=105.5211, 90th=127.2642, 95th=142.8774
    After mean+0.3*std threshold (96.5431): 524/1139 predictions
    Model predictions: 524, GT intersections: 954
    TP=216, FP=308, FN=0
    TP=216, FP=308, FN=0
    Precision=0.4122, Recall=1.0000, F1=0.5838
  2x2y1z: 3272 graphs after filter (min_nodes=10)
  Training: 1 epochs, batch_size=32, lr=0.01


olympic:  36%|███▋      | 4/11 [02:59<05:14, 44.93s/it]


KeyboardInterrupt: 

In [ ]:
# Cell 9: Final Summary Tables & Save to CSV
# ============================================================
# GT = Intersection voxels (where 2+ channels overlap)
# Compare ALL model predictions with ALL GT (no filtering)
# ============================================================

# Include all textures in summary
TEXTURES_FOR_SUMMARY = TEXTURES
print(f"Textures included in summary: {TEXTURES_FOR_SUMMARY}")

# Collect all results
combined = []
for tex, res_list in all_results.items():
    for r in res_list:
        r_copy = r.copy()
        r_copy['Texture'] = tex
        combined.append(r_copy)

print(f"Total results: {len(combined)}")

if combined:
    # ============================================================
    # 1. Save each texture results to separate CSV
    # ============================================================
    print(f"\n{'='*80}")
    print("SAVING INDIVIDUAL TEXTURE RESULTS")
    print(f"{'='*80}")
    
    for tex in TEXTURES_FOR_SUMMARY:
        tex_data = [r for r in combined if r['Texture'] == tex]
        if tex_data:
            tex_df = pd.DataFrame(tex_data)
            tex_csv_path = os.path.join(BASE_DIR, f'results_{tex}.csv')
            tex_df.to_csv(tex_csv_path, index=False)
            print(f"  Saved: {tex_csv_path} ({len(tex_data)} rows)")
    
    # ============================================================
    # 2. Create Average by Texture table and save to CSV
    # ============================================================
    print(f"\n{'='*100}")
 
    print(f"{'Texture':>12} {'Exp':>8} {'Avg TP':>8} {'Avg FP':>8} {'Avg FN':>8} {'Avg Prec':>10} {'Avg Rec':>8} {'Avg F1':>8} {'Avg Acc':>10}")
    print("-"*100)
    
    avg_by_texture = []
    for tex in TEXTURES_FOR_SUMMARY:
        tex_data = [r for r in combined if r['Texture'] == tex]
        if tex_data:
            avg_row = {
                'Texture': tex,
                'Experiments': len(tex_data),
                'Avg TP': np.mean([r['TP'] for r in tex_data]),
                'Avg FP': np.mean([r['FP'] for r in tex_data]),
                'Avg FN': np.mean([r['FN'] for r in tex_data]),
                'Avg Precision': np.mean([r['Precision'] for r in tex_data]),
                'Avg Recall': np.mean([r['Recall'] for r in tex_data]),
                'Avg F1 Score': np.mean([r['F1 Score'] for r in tex_data]),
                'Avg Accuracy': np.mean([r['Accuracy'] for r in tex_data])
            }
            avg_by_texture.append(avg_row)
            print(f"{tex:>12} {avg_row['Experiments']:>8} {avg_row['Avg TP']:>8.1f} {avg_row['Avg FP']:>8.1f} {avg_row['Avg FN']:>8.1f} {avg_row['Avg Precision']:>10.4f} {avg_row['Avg Recall']:>8.4f} {avg_row['Avg F1 Score']:>8.4f} {avg_row['Avg Accuracy']:>10.4f}")
    
    print("="*100)
    
    # Save average by texture
    avg_texture_df = pd.DataFrame(avg_by_texture)
    avg_texture_csv = os.path.join(BASE_DIR, 'average_by_texture.csv')
    avg_texture_df.to_csv(avg_texture_csv, index=False)
    print(f"  Saved: {avg_texture_csv}")
    
    # ============================================================
    # 3. Create Average by Scale table and save to CSV
    # ============================================================
    print(f"\n{'='*100}")

    print(f"{'Scale':>12} {'Config':>14} {'Exp':>8} {'Avg TP':>8} {'Avg FP':>8} {'Avg FN':>8} {'Avg Prec':>10} {'Avg Rec':>8} {'Avg F1':>8}")
    print("-"*100)
    
    avg_by_scale = []
    for scale_name, x_s, y_s, z_s in SCALES:
        scale_data = [r for r in combined if r['Scale Name'] == scale_name]
        if scale_data:
            avg_row = {
                'Scale Name': scale_name,
                'Config': f'{x_s}x,{y_s}y,{z_s}z',
                'Experiments': len(scale_data),
                'Avg TP': np.mean([r['TP'] for r in scale_data]),
                'Avg FP': np.mean([r['FP'] for r in scale_data]),
                'Avg FN': np.mean([r['FN'] for r in scale_data]),
                'Avg Precision': np.mean([r['Precision'] for r in scale_data]),
                'Avg Recall': np.mean([r['Recall'] for r in scale_data]),
                'Avg F1 Score': np.mean([r['F1 Score'] for r in scale_data]),
                'Avg Accuracy': np.mean([r['Accuracy'] for r in scale_data])
            }
            avg_by_scale.append(avg_row)
            print(f"{scale_name:>12} {avg_row['Config']:>14} {avg_row['Experiments']:>8} {avg_row['Avg TP']:>8.1f} {avg_row['Avg FP']:>8.1f} {avg_row['Avg FN']:>8.1f} {avg_row['Avg Precision']:>10.4f} {avg_row['Avg Recall']:>8.4f} {avg_row['Avg F1 Score']:>8.4f}")
    
    print("="*100)
    
    # Save average by scale
    avg_scale_df = pd.DataFrame(avg_by_scale)
    avg_scale_csv = os.path.join(BASE_DIR, 'average_by_scale.csv')
    avg_scale_df.to_csv(avg_scale_csv, index=False)
    print(f"  Saved: {avg_scale_csv}")
    
    # ============================================================
    # 4. Overall Average and save to CSV
    # ============================================================
    print(f"\n{'='*100}")
    print("OVERALL AVERAGE (all experiments)")
    print("-"*100)
    
    if combined:
        overall_avg = {
            'Total Experiments': len(combined),
            'Overall Avg TP': np.mean([r['TP'] for r in combined]),
            'Overall Avg FP': np.mean([r['FP'] for r in combined]),
            'Overall Avg FN': np.mean([r['FN'] for r in combined]),
            'Overall Avg Precision': np.mean([r['Precision'] for r in combined]),
            'Overall Avg Recall': np.mean([r['Recall'] for r in combined]),
            'Overall Avg F1 Score': np.mean([r['F1 Score'] for r in combined]),
            'Overall Avg Accuracy': np.mean([r['Accuracy'] for r in combined])
        }
        
        print(f"  Total Experiments: {overall_avg['Total Experiments']}")
        print(f"  Avg TP: {overall_avg['Overall Avg TP']:.2f}")
        print(f"  Avg FP: {overall_avg['Overall Avg FP']:.2f}")
        print(f"  Avg FN: {overall_avg['Overall Avg FN']:.2f}")
        print(f"  Avg Precision: {overall_avg['Overall Avg Precision']:.4f}")
        print(f"  Avg Recall: {overall_avg['Overall Avg Recall']:.4f}")
        print(f"  Avg F1 Score: {overall_avg['Overall Avg F1 Score']:.4f}")
        print(f"  Avg Accuracy (IoU): {overall_avg['Overall Avg Accuracy']:.4f}")
    else:
        print(f"  No results")
        overall_avg = {}
    
    print("="*100)
    
    # Save overall average
    overall_df = pd.DataFrame([overall_avg])
    overall_csv = os.path.join(BASE_DIR, 'overall_average.csv')
    overall_df.to_csv(overall_csv, index=False)
    print(f"  Saved: {overall_csv}")
    
    # ============================================================
    # 5. Save all combined results
    # ============================================================
    all_results_csv = os.path.join(BASE_DIR, 'all_textures_all_scales_results.csv')
    pd.DataFrame(combined).to_csv(all_results_csv, index=False)
    print(f"  Saved: {all_results_csv}")

# ============================================================
# Summary of saved files
# ============================================================
print(f"\n{'='*80}")
print("ALL CSV FILES SAVED:")
print(f"{'='*80}")
print(f"  Directory: {BASE_DIR}")
print(f"  - results_<texture>.csv  : Results for each texture (5 files)")
print(f"  - average_by_texture.csv : Average metrics per texture")
print(f"  - overall_average.csv    : Overall average metrics")
print(f"  - all_textures_all_scales_results.csv : All raw results")
print(f"{'='*80}")
print("EXPERIMENT COMPLETE!")
print(f"{'='*80}")

======================================================================================================
 Scale Name   Scale       Model     GT      TP    FP        FN   Precision  Recall   F1     Acc -IOU
------------------------------------------------------------------------------------------------------
 original   1x,1x,1x      920.7   768.0   768.0   152.7     0.0  0.8341   1.0000   0.9096   0.8341
 2x1y1z     2x,1x,1x     1714.6  1265.0  1265.0   449.6     0.0  0.7378   1.0000   0.8491   0.7378
 1x2y1z     1x,2x,1x     1641.1  1424.7  1424.7   216.4   216.4  0.8681   0.8681   0.9294   0.8681
 1x1y2z     1x,1x,2x      841.0   710.3   710.3   130.7   130.7  0.8446   0.8446   0.9158   0.8446
 2x2y1z     2x,2x,1x     2544.1  2163.0  2163.0   381.1   381.1  0.8502   0.8502   0.9190   0.8502
 1x2y2z     1x,2x,2x     1868.6  1509.7  1509.7   358.9   358.9  0.8080   0.8080   0.8938   0.8080
 2x2y2z     2x,2x,2x     5106.2  2857.3  2857.3  2248.9  2248.9  0.5596   0.5596   0.7176   0.5596
 3x1y1z     3x,1x,1x     3628.1  2019.0  2019.0  1609.1  1609.1  0.5565   0.5565   0.7151   0.5565
 3x2y1z     3x,2x,1x     5373.7  3588.0  3588.0  1785.7  1785.7  0.6677   0.6677   0.8007   0.6677
 3x1y2z     3x,1x,2x     3646.5  2051.3  2051.3  1595.2  1595.2  0.5625   0.5625   0.7200   0.5625
 3x2y2z     3x,2x,2x     5821.6  3604.7  3604.7  2216.9  2216.9  0.6192   0.6192   0.7648   0.6192
====================================================================================================




======================================================================================================
 Scale Name   Scale       Model     GT      TP    FP        FN   Precision  Recall   F1     
------------------------------------------------------------------------------------------------------
 original   1x,1x,1x      920.7   768.0   768.0   152.7     0.0  0.8341   1.0000   0.9096   
 2x1y1z     2x,1x,1x     1714.6  1265.0  1265.0   449.6     0.0  0.7378   1.0000   0.8491   
 1x2y1z     1x,2x,1x     1641.1  1424.7  1424.7   216.4   216.4  0.8681   0.8681   0.9294   
 1x1y2z     1x,1x,2x      841.0   710.3   710.3   130.7   130.7  0.8446   0.8446   0.9158   
 2x2y1z     2x,2x,1x     2544.1  2163.0  2163.0   381.1   381.1  0.8502   0.8502   0.9190   
 1x2y2z     1x,2x,2x     1868.6  1509.7  1509.7   358.9   358.9  0.8080   0.8080   0.8938   
 2x2y2z     2x,2x,2x     5106.2  2857.3  2857.3  2248.9  2248.9  0.5596   0.5596   0.7176   
 3x1y1z     3x,1x,1x     3628.1  2019.0  2019.0  1609.1  1609.1  0.5565   0.5565   0.7151   
 3x2y1z     3x,2x,1x     5373.7  3588.0  3588.0  1785.7  1785.7  0.6677   0.6677   0.8007   
 3x1y2z     3x,1x,2x     3646.5  2051.3  2051.3  1595.2  1595.2  0.5625   0.5625   0.7200   
 3x2y2z     3x,2x,2x     5821.6  3604.7  3604.7  2216.9  2216.9  0.6192   0.6192   0.7648   
====================================================================================================